# Notebook 4 – Duplicate Data Handling

## 1. What are duplicate records?
Duplicate records are rows that repeat all or part of the information in another row. They can be caused by repeated imports, retries, manual entry, or legitimate repeated events.

## 2. Exact vs. partial duplicates
- **Exact duplicate:** every selected column has the same value.
- **Partial duplicate:** some important fields match, but other fields differ. For example, the same customer email may appear with a different phone number.

A repeated value is not automatically a duplicate record. Two purchases by the same customer may be two valid transactions.

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [101, 102, 102, 103, 104, 104],
    "name": ["Anu", "Bala", "Bala", "Charan", "Divya", "Divya"],
    "email": ["anu@example.com", "bala@example.com", "bala@example.com",
              "charan@example.com", "divya@example.com", "divya.new@example.com"],
    "city": ["Chennai", "Salem", "Salem", "Madurai", "Trichy", "Trichy"]
})
customers

## 3. Identifying duplicates with `duplicated()`
`duplicated()` returns `True` for rows considered duplicates. By default, the first occurrence is marked `False` and later occurrences are `True`.

- `keep='first'`: keep the first occurrence.
- `keep='last'`: keep the last occurrence.
- `keep=False`: mark every row in a duplicate group.

In [ ]:
# Exact duplicate rows across all columns
customers.duplicated()

# Show the repeated rows
customers[customers.duplicated(keep=False)]

## 4. Duplicate detection using selected columns
Use `subset` when business identity depends on particular columns. For example, customer email may be used as a candidate identifier, but first verify that email is expected to be unique.

In [ ]:
# Repeated customer IDs
customers[customers.duplicated(subset=["customer_id"], keep=False)]

# Repeated email addresses
customers[customers.duplicated(subset=["email"], keep=False)]

## 5. Removing duplicates with `drop_duplicates()`
`drop_duplicates()` returns a DataFrame with duplicates removed. Use `subset` to define what counts as a duplicate and `keep` to choose which row to retain.

In [ ]:
# Remove exact duplicate rows only
exact_deduped = customers.drop_duplicates()
exact_deduped

In [ ]:
# Example: keep the last row for each customer ID.
# This is appropriate only if later rows are known to be the most current.
latest_by_id = customers.drop_duplicates(subset=["customer_id"], keep="last")
latest_by_id

## 6. Why blindly deleting duplicates can be dangerous
1. **Legitimate repeated events:** two identical-looking orders can both be real.
2. **Wrong identity key:** two people can share a name; one person can have multiple phone numbers.
3. **Loss of updates:** keeping the first row may discard newer information.
4. **Audit and financial impact:** removing a valid transaction can change revenue, balances, or compliance records.
5. **Data integration problems:** repeated rows may indicate an upstream pipeline issue that should be fixed at its source.

## 7. A safer business-rule workflow
1. Define what makes a record unique (for example, transaction ID).
2. Inspect duplicate groups before changing data.
3. Decide which row to retain using a documented rule (latest timestamp, verified source, etc.).
4. Preserve the raw data and log removed records.
5. Compare row counts and business totals before and after cleaning.

In [ ]:
# Example business rule: transaction_id should be unique.
transactions = pd.DataFrame({
    "transaction_id": ["T01", "T02", "T02", "T03"],
    "customer_id": [10, 11, 11, 12],
    "amount": [250, 500, 500, 120],
    "created_at": pd.to_datetime(["2026-10-01", "2026-10-02",
                                  "2026-10-02", "2026-10-03"])
})

duplicate_transactions = transactions[
    transactions.duplicated(subset=["transaction_id"], keep=False)
]
print("Duplicate transaction groups:")
display(duplicate_transactions)

# Do not remove automatically until the transaction rule is verified.
print("Rows before review:", len(transactions))
print("Total amount before review:", transactions["amount"].sum())